# Summary of Trigger Proposal

In [110]:
%load_ext jupyter_black
%load_ext autoreload
%autoreload 2

The jupyter_black extension is already loaded. To reload it, use:
  %reload_ext jupyter_black
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [111]:
import ocha_stratus as stratus
from src.datasources.era5 import fetch_era5_data
from src.datasources.seas5 import fetch_seas5_data
from src.constants import *
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, ScalarFormatter
from affine import Affine
from rasterstats import zonal_stats
import calendar
import math

In [112]:
pd.options.display.float_format = "{:,.1f}".format

In [113]:
mam_blob_name = "ds-aa-eth-drought/exploration/Ethiopia OND zones.csv"
start_date = pd.Timestamp("1997-01-01")
end_date = pd.Timestamp("2025-12-01")
season_months = [10, 11, 12]
issued_months = [6, 7, 8, 9]

In [114]:
# reading in population data
eth_adm_pop = stratus.load_csv_from_blob(
    "ds-aa-eth-drought/processed/eth_adm2_pop.csv",
    stage="dev",
    container_name="projects",
)
validation_data = (
    "ds-aa-eth-drought/exploration/Ethiopia Drought Validation Data OND.csv"
)
validation_csv = stratus.load_csv_from_blob(
    validation_data, stage="dev", container_name="projects"
)
validation_csv["year"] = validation_csv["Season"].str[-4:].astype(int)

In [115]:
season_csv = stratus.load_csv_from_blob(
    mam_blob_name, stage="dev", container_name="projects"
)
seas5_data = fetch_seas5_data(
    season_csv["adm2_pcode"].unique(), start_date=start_date, end_date=end_date
)

era5_data = fetch_era5_data(
    season_csv["adm2_pcode"].unique(), start_date=start_date, end_date=end_date
)
era5_data["year"] = pd.to_datetime(era5_data["valid_date"]).dt.year
era5_data["valid_month"] = pd.to_datetime(era5_data["valid_date"]).dt.month
era5_data["mean"] = era5_data["mean"] * era5_data["valid_month"].map(days_in_month)

In [116]:
seas5_df = seas5_data.copy()
seas5_df["issued_date"] = pd.to_datetime(seas5_df["issued_date"])
seas5_df["valid_date"] = pd.to_datetime(seas5_df["valid_date"])

seas5_df["issued_month"] = seas5_df["issued_date"].dt.month
seas5_df["valid_month"] = seas5_df["valid_date"].dt.month
seas5_df["year"] = seas5_df["valid_date"].dt.year
seas5_df["mean"] = seas5_df["mean"] * seas5_df["valid_month"].map(days_in_month)

seas5_df = (
    seas5_df[
        (seas5_df["issued_month"].isin(issued_months))
        & (seas5_df["valid_month"].isin(season_months))
    ]
    .groupby(["year", "pcode", "issued_month"], as_index=False)["mean"]
    .sum()
)
seas5_df["rank"] = seas5_df.groupby(["pcode", "issued_month"])["mean"].rank(
    method="min", ascending=True
)
n_years = seas5_df["year"].nunique()
all_years = seas5_df["year"].unique()
seas5_df["return_period"] = ((n_years + 1) / seas5_df["rank"]).round(1)
df_rp = seas5_df[seas5_df["return_period"] >= 5].copy()
drought_counts = (
    df_rp.groupby(["issued_month", "year"])
    .size()
    .reset_index(name="Predictive")
    .pivot(index="year", columns="issued_month", values="Predictive")
    .fillna(0)
    .astype(int)
    .sort_index()
)

drought_counts.columns = [calendar.month_abbr[m] for m in drought_counts.columns]
drought_counts.head()

,Jun,Jul,Aug,Sep
year,,,,
1998,0,4,5,15
1999,2,0,10,11
2000,2,19,17,10
2001,4,0,10,8
2003,0,0,1,3


In [117]:
rangeland_wrsi = "ds-aa-eth-drought/exploration/leap/Rangeland WRSI.csv"
wrsi_csv = stratus.load_csv_from_blob(
    rangeland_wrsi, stage="dev", container_name="projects"
)
# clean up
wrsi_csv = wrsi_csv.dropna(how="all")
wrsi_csv = wrsi_csv.rename(columns={"Name": "region", "Unnamed: 1": "zone"}).dropna(
    subset=["region", "zone"]
)
# only keep regions in Somali or snnpr or if ormomia, zones buji and borena
wrsi_csv = wrsi_csv[
    wrsi_csv["region"].str.contains("Somali|SNNPR")
    | (
        wrsi_csv["region"].str.contains("Oromia")
        & wrsi_csv["zone"].str.contains("Borena|Guji")
    )
]
dk3_cols = [c for c in wrsi_csv.columns if c.endswith("_10_3")]
dk3 = wrsi_csv[["region", "zone"] + dk3_cols]

# long format
dk3_long = dk3.melt(
    id_vars=["region", "zone"], var_name="year_dekad", value_name="wrsi"
)

# extract year
dk3_long["year"] = dk3_long["year_dekad"].str.split("_").str[0].astype(int)

# drop missing WRSI
dk3_long = dk3_long.dropna(subset=["wrsi"])


def classify_wrsi(x):
    if x <= 20:
        return "Very poor"
    elif x <= 40:
        return "Poor"
    elif x <= 60:
        return "Average"
    elif x <= 80:
        return "Good"
    else:
        return "Very good"


dk3_long["wrsi_class"] = dk3_long["wrsi"].apply(classify_wrsi)
zone_counts = (
    dk3_long.groupby(["year", "wrsi_class"])
    .size()
    .reset_index(name="n_zones")
    .sort_values(["year", "wrsi_class"])
)
zones_per_year = dk3_long.groupby("year")["zone"].nunique()
class_order = ["Very poor", "Poor", "Average", "Good", "Very good"]
zone_counts_wide = (
    zone_counts.assign(
        wrsi_class=lambda d: pd.Categorical(d["wrsi_class"], class_order)
    )
    .pivot(index="year", columns="wrsi_class", values="n_zones")
    .fillna(0)
    .sort_index()
)
# total poor + very poor zones per year
oct_rangeland_poor_vpoor = (
    zone_counts_wide[["Very poor", "Poor"]]
    .sum(axis=1)
    .reset_index(name="Rangeland WRSI (Oct)")
)

In [118]:
rangeland_wrsi = "ds-aa-eth-drought/exploration/leap/Rangeland WRSI.csv"
wrsi_csv = stratus.load_csv_from_blob(
    rangeland_wrsi, stage="dev", container_name="projects"
)
# keep only those admin2pcode in the season_csv

# clean up
wrsi_csv = wrsi_csv.dropna(how="all")
wrsi_csv = wrsi_csv.rename(columns={"Name": "region", "Unnamed: 1": "zone"}).dropna(
    subset=["region", "zone"]
)
wrsi_csv = wrsi_csv[
    wrsi_csv["region"].str.contains("Somali|SNNPR")
    | (
        wrsi_csv["region"].str.contains("Oromia")
        & wrsi_csv["zone"].str.contains("Borena|Guji")
    )
]
dk3_cols = [c for c in wrsi_csv.columns if c.endswith("_11_3")]
dk3 = wrsi_csv[["region", "zone"] + dk3_cols]

# long format
dk3_long = dk3.melt(
    id_vars=["region", "zone"], var_name="year_dekad", value_name="wrsi"
)

# extract year
dk3_long["year"] = dk3_long["year_dekad"].str.split("_").str[0].astype(int)

# drop missing WRSI
dk3_long = dk3_long.dropna(subset=["wrsi"])


def classify_wrsi(x):
    if x <= 20:
        return "Very poor"
    elif x <= 40:
        return "Poor"
    elif x <= 60:
        return "Average"
    elif x <= 80:
        return "Good"
    else:
        return "Very good"


dk3_long["wrsi_class"] = dk3_long["wrsi"].apply(classify_wrsi)
zone_counts = (
    dk3_long.groupby(["year", "wrsi_class"])
    .size()
    .reset_index(name="n_zones")
    .sort_values(["year", "wrsi_class"])
)
zones_per_year = dk3_long.groupby("year")["zone"].nunique()
class_order = ["Very poor", "Poor", "Average", "Good", "Very good"]
zone_counts_wide = (
    zone_counts.assign(
        wrsi_class=lambda d: pd.Categorical(d["wrsi_class"], class_order)
    )
    .pivot(index="year", columns="wrsi_class", values="n_zones")
    .fillna(0)
    .sort_index()
)
# total poor + very poor zones per year
nov_rangeland_poor_vpoor = (
    zone_counts_wide[["Very poor", "Poor"]]
    .sum(axis=1)
    .reset_index(name="Rangeland WRSI (Nov)")
)

In [119]:
rangeland_wrsi = "ds-aa-eth-drought/exploration/leap/Rangeland WRSI.csv"
wrsi_csv = stratus.load_csv_from_blob(
    rangeland_wrsi, stage="dev", container_name="projects"
)
# keep only those admin2pcode in the season_csv

# clean up
wrsi_csv = wrsi_csv.dropna(how="all")
wrsi_csv = wrsi_csv.rename(columns={"Name": "region", "Unnamed: 1": "zone"}).dropna(
    subset=["region", "zone"]
)
wrsi_csv = wrsi_csv[
    wrsi_csv["region"].str.contains("Somali|SNNPR")
    | (
        wrsi_csv["region"].str.contains("Oromia")
        & wrsi_csv["zone"].str.contains("Borena|Guji")
    )
]
dk3_cols = [c for c in wrsi_csv.columns if c.endswith("_12_3")]
dk3 = wrsi_csv[["region", "zone"] + dk3_cols]

# long format
dk3_long = dk3.melt(
    id_vars=["region", "zone"], var_name="year_dekad", value_name="wrsi"
)

# extract year
dk3_long["year"] = dk3_long["year_dekad"].str.split("_").str[0].astype(int)

# drop missing WRSI
dk3_long = dk3_long.dropna(subset=["wrsi"])


def classify_wrsi(x):
    if x <= 20:
        return "Very poor"
    elif x <= 40:
        return "Poor"
    elif x <= 60:
        return "Average"
    elif x <= 80:
        return "Good"
    else:
        return "Very good"


dk3_long["wrsi_class"] = dk3_long["wrsi"].apply(classify_wrsi)
zone_counts = (
    dk3_long.groupby(["year", "wrsi_class"])
    .size()
    .reset_index(name="n_zones")
    .sort_values(["year", "wrsi_class"])
)
zones_per_year = dk3_long.groupby("year")["zone"].nunique()
class_order = ["Very poor", "Poor", "Average", "Good", "Very good"]
zone_counts_wide = (
    zone_counts.assign(
        wrsi_class=lambda d: pd.Categorical(d["wrsi_class"], class_order)
    )
    .pivot(index="year", columns="wrsi_class", values="n_zones")
    .fillna(0)
    .sort_index()
)
# total poor + very poor zones per year
seas_rangeland_poor_vpoor = (
    zone_counts_wide[["Very poor", "Poor"]]
    .sum(axis=1)
    .reset_index(name="Rangeland WRSI (Dec)")
)

In [120]:
era_zone_year = (
    era5_data[era5_data["valid_month"].isin([10])]
    .groupby(["pcode", "year"])["mean"]
    .sum()
    .reset_index()
)

era_zone_year["rank"] = era_zone_year.groupby("pcode")["mean"].rank(
    method="min", ascending=True
)
n_years = era_zone_year["year"].nunique()
era_zone_year["return_period"] = ((n_years + 1) / era_zone_year["rank"]).round(1)

# merge population at zone level
era_zone_year = era_zone_year.merge(
    eth_adm_pop[["adm2_src", "sum"]],
    left_on="pcode",
    right_on="adm2_src",
    how="left",
)

# sum population for zones crossing 1-in-5 RP (October)
era5_zones_oct_pop = (
    era_zone_year[era_zone_year["return_period"] >= 5]
    .groupby("year")  # ["sum"]
    .size()
    .reset_index(name="Affected Zones (Oct)")
)

In [121]:
era_zone_year = (
    era5_data[era5_data["valid_month"].isin([10, 11])]
    .groupby(["pcode", "year"])["mean"]
    .sum()
    .reset_index()
)

era_zone_year["rank"] = era_zone_year.groupby("pcode")["mean"].rank(
    method="min", ascending=True
)
n_years = era_zone_year["year"].nunique()
era_zone_year["return_period"] = ((n_years + 1) / era_zone_year["rank"]).round(1)


# sum population for zones crossing 1-in-5 RP
era5_zones_nov_pop = (
    era_zone_year[era_zone_year["return_period"] >= 5]
    .groupby("year")  # ["sum"]
    .size()
    .reset_index(name="Affected Zones (Nov)")
)

In [122]:
era_zone_year = (
    era5_data[era5_data["valid_month"].isin([10, 11, 12])]
    .groupby(["pcode", "year"])["mean"]
    .sum()
    .reset_index()
)

era_zone_year["rank"] = era_zone_year.groupby("pcode")["mean"].rank(
    method="min", ascending=True
)
n_years = era_zone_year["year"].nunique()
era_zone_year["return_period"] = ((n_years + 1) / era_zone_year["rank"]).round(1)


# sum population for zones crossing 1-in-5 RP
era5_zones_seas_pop = (
    era_zone_year[era_zone_year["return_period"] >= 5]
    .groupby("year")  # ["sum"]
    .size()
    .reset_index(name="Zones (Observed)")
)

In [123]:
# vhi and asi data
vhi_asi_data = stratus.load_csv_from_blob(
    "ds-aa-eth-drought/processed/eth_asi_mam_jjas_ond.csv",
    stage="dev",
    container_name="projects",
)

In [151]:
era5_seasonal = (
    era5_data[era5_data["valid_month"].isin(season_months)]
    .groupby(["year", "pcode", "valid_month"], as_index=False)["mean"]
    .mean()
    .groupby(["year", "pcode"], as_index=False)["mean"]
    .sum()
    .groupby("year", as_index=False)["mean"]
    .mean()
    .rename(columns={"mean": "ERA5 Seasonal Rainfall"})
    .sort_values("ERA5 Seasonal Rainfall", ascending=True)
    .reset_index(drop=True)
)
# era5_seasonal = era5_seasonal.merge(drought_counts, on="year", how="left")
era5_seasonal = era5_seasonal.merge(oct_rangeland_poor_vpoor, on="year", how="left")
era5_seasonal = era5_seasonal.merge(nov_rangeland_poor_vpoor, on="year", how="left")
era5_seasonal = era5_seasonal.merge(era5_zones_oct_pop, on="year", how="left")
era5_seasonal = era5_seasonal.merge(era5_zones_nov_pop, on="year", how="left")
era5_seasonal = era5_seasonal.merge(era5_zones_seas_pop, on="year", how="left")
era5_seasonal = era5_seasonal.merge(seas_rangeland_poor_vpoor, on="year", how="left")

era5_seasonal = era5_seasonal.merge(
    validation_csv.drop(columns=["Season"]),
    left_on="year",
    right_on="year",
    how="left",
)

asi_cols = ["year"] + [
    c for c in vhi_asi_data.columns if c.lower().startswith("ond_asi_wavg")
]

era5_seasonal = era5_seasonal.merge(
    vhi_asi_data[asi_cols],
    on="year",
    how="left",
)
era5_seasonal = era5_seasonal.rename(
    columns=lambda c: (
        c.lower().replace("ond_", "").replace("_", " ").upper()
        if c.lower().startswith("ond_")
        else c
    )
)
# drop column mar and may
# era5_seasonal = era5_seasonal.drop(columns=["Jun", "Aug"])
era5_seasonal = era5_seasonal.rename(
    columns={
        "year": "Year",
        "ERA5 Seasonal Rainfall": "ERA5 Seasonal Rainfall (mm)",
        "Jun": "SEAS5 Jun Forecast (Number of Zones ≥ 5 Yr RP)",
        "Jul": "SEAS5 Jul Forecast (Number of Zones ≥ 5 Yr RP)",
        "Aug": "SEAS5 Aug Forecast (Number of Zones ≥ 5 Yr RP)",
        "Sep": "SEAS5 Sep Forecast (Number of Zones ≥ 5 Yr RP)",
        "Affected Zones (Oct)": "ERA5 Oct (Number of Zones ≥ 5 Yr RP)",
        "Rangeland WRSI (Oct)": "Rangeland WRSI (Oct, poor+very poor)",
        "Affected Zones (Nov)": "ERA5 Nov (Number of Zones ≥ 5 Yr RP)",
        "Rangeland WRSI (Nov)": "Rangeland WRSI (Nov, poor+very poor)",
        "Rangeland WRSI (Dec)": "Rangeland WRSI (Dec, poor+very poor)",
        "Zones (Observed)": "Zones ≥ 5-Year RP (Observed)",
        "CERF Allocations": "CERF Allocations (USD)",
        "People Affected": "People Affected",
        "ASI WAVG": "ASI",
    }
)
HIGHLIGHT_RULES = {}

for col in era5_seasonal.columns:
    if col.startswith("SEAS5 "):
        HIGHLIGHT_RULES[col] = (14, "#78A2D8")

    elif col.startswith("ERA5 Oct"):
        HIGHLIGHT_RULES[col] = (7, "#DABF86")

    elif col.startswith("ERA5 Nov"):
        HIGHLIGHT_RULES[col] = (9, "#DABF86")

    elif col.startswith(
        ("Rangeland WRSI (Oct, poor+very poor)", "Rangeland WRSI (Nov, poor+very poor)")
    ):
        HIGHLIGHT_RULES[col] = (12, "#DABF86")

    elif col.startswith("Rangeland WRSI (Dec, poor+very poor)"):
        HIGHLIGHT_RULES[col] = (13, "#72D28F")

    elif col.startswith(("Cropland WRSI", "Zones ≥")):
        HIGHLIGHT_RULES[col] = (8, "#72D28F")

    elif col.startswith("ASI"):
        HIGHLIGHT_RULES[col] = (35, "#D4782C")

In [152]:
styled_table = era5_seasonal.style.format(
    {
        c: ("{:,.1f}" if c.lower().startswith(("asi", "vhi")) else "{:,.0f}")
        for c in era5_seasonal.columns
        if (pd.api.types.is_numeric_dtype(era5_seasonal[c]) and c != "Year")
    },
    na_rep="",
)

for col, (thr, color) in HIGHLIGHT_RULES.items():
    styled_table = styled_table.map(
        lambda v, t=thr, c=color: (
            f"background-color: {c};" if pd.notna(v) and v >= t else ""
        ),
        subset=[col],
    )
styled_table

,Year,ERA5 Seasonal Rainfall (mm),"Rangeland WRSI (Oct, poor+very poor)","Rangeland WRSI (Nov, poor+very poor)",ERA5 Oct (Number of Zones ≥ 5 Yr RP),ERA5 Nov (Number of Zones ≥ 5 Yr RP),Zones ≥ 5-Year RP (Observed),"Rangeland WRSI (Dec, poor+very poor)",CERF Allocations (USD),People Affected,ASI
0,2021,72,12,12,17,18,19,12,"4,987,750","6,800,000",17.4
1,2010,85,10,13,10,19,19,13,,,1.6
2,2022,95,12,12,7,14,17,12,,"24,100,000",17.6
3,1998,104,6,10,10,11,15,13,,,0.0
4,2025,111,9,,9,11,11,,,,21.5
5,2016,121,12,11,18,8,9,13,"18,512,690",,2.7
6,2007,131,11,11,6,6,8,13,"19,999,594",,2.7
7,1999,134,8,9,4,9,6,13,,,1.6
8,2005,144,12,12,,1,2,13,"3,978,239","2,600,000",5.8
9,2003,156,12,12,18,10,4,12,,"12,600,000",0.1


In [153]:
trigger_mask = pd.Series(False, index=era5_seasonal.index)

for col, (thr, _) in HIGHLIGHT_RULES.items():
    if col in era5_seasonal.columns:
        trigger_mask |= pd.to_numeric(era5_seasonal[col], errors="coerce") >= thr
styled_table = styled_table.apply(
    lambda s: ["font-weight: bold;" if trigger_mask.loc[i] else "" for i in s.index],
    axis=0,
    # subset=["Year"],
)
styled_table

,Year,ERA5 Seasonal Rainfall (mm),"Rangeland WRSI (Oct, poor+very poor)","Rangeland WRSI (Nov, poor+very poor)",ERA5 Oct (Number of Zones ≥ 5 Yr RP),ERA5 Nov (Number of Zones ≥ 5 Yr RP),Zones ≥ 5-Year RP (Observed),"Rangeland WRSI (Dec, poor+very poor)",CERF Allocations (USD),People Affected,ASI
0,2021,72,12,12,17,18,19,12,"4,987,750","6,800,000",17.4
1,2010,85,10,13,10,19,19,13,,,1.6
2,2022,95,12,12,7,14,17,12,,"24,100,000",17.6
3,1998,104,6,10,10,11,15,13,,,0.0
4,2025,111,9,,9,11,11,,,,21.5
5,2016,121,12,11,18,8,9,13,"18,512,690",,2.7
6,2007,131,11,11,6,6,8,13,"19,999,594",,2.7
7,1999,134,8,9,4,9,6,13,,,1.6
8,2005,144,12,12,,1,2,13,"3,978,239","2,600,000",5.8
9,2003,156,12,12,18,10,4,12,,"12,600,000",0.1
